## Problema de cobertura (conjunto dominante)

In [2]:
%pip install pulp -q

import pulp

prob = pulp.LpProblem("Cobertura", pulp.LpMinimize)

# ENUNCIADO
# O governo planeja construir escolas de modo a satisfazer a demanda em uma cidade.
# A lei demanda que todo bairro deve ter uma escola ou estar proximo de uma.
# Objetivo: decidir em quais bairros construir escolas para atender a lei e
# minimizar o numero de construcoes.

# ENTRADA: grafo simples G(V, E)
V = [1, 2, 3, 4, 5, 6]
E = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6)]

# Vizinhanca N(i) de cada vertice
N = {i: set() for i in V}
for u, v in E:
  N[u].add(v)
  N[v].add(u)

# VARIAVEIS DE DECISAO
# x_i = 1 se o vertice i pertence a cobertura (escola no bairro i), 0 caso contrario
x = {i: pulp.LpVariable(f"x_{i}", cat='Binary') for i in V}

## RESTRICOES
# (dominio) x_i pertence a {0, 1} — garantido por cat='Binary'

# (cobertura) x_i + somatorio de x_j, j pertencente a N(i) >= 1, para todo i em V
for i in V:
  prob += x[i] + pulp.lpSum(x[j] for j in N[i]) >= 1

# FUNCAO OBJETIVO: min somatorio de x_i, i pertencente a V
prob += pulp.lpSum(x[i] for i in V)

prob.solve(pulp.PULP_CBC_CMD(msg=False))

# Resultados
escolha = {i: x[i].varValue for i in V}
total_escolas = int(pulp.value(prob.objective))
bairros_com_escola = [i for i in V if escolha[i] == 1]

print("\nPROBLEMA DE COBERTURA (CONJUNTO DOMINANTE)\n")
print(f"Numero minimo de escolas: {total_escolas}\n")
print("Construir escolas nos bairros:", bairros_com_escola)

print("\nVerificacao de cobertura por bairro:")
print("Bairro\tEscola aqui?\tCoberto por escola em\tAtendido?")
for i in V:
  tem_escola = 'sim' if escolha[i] == 1 else 'nao'
  cobridores = [j for j in ([i] + sorted(N[i])) if escolha[j] == 1]
  status = 'sim' if cobridores else 'nao'
  print(f"B{i}\t{tem_escola}\t\t{', '.join(f'B{j}' for j in cobridores) if cobridores else '-'}\t\t{status}")

Note: you may need to restart the kernel to use updated packages.

PROBLEMA DE COBERTURA (CONJUNTO DOMINANTE)

Numero minimo de escolas: 2

Construir escolas nos bairros: [2, 5]

Verificacao de cobertura por bairro:
Bairro	Escola aqui?	Coberto por escola em	Atendido?
B1	nao		B2		sim
B2	sim		B2		sim
B3	nao		B2		sim
B4	nao		B5		sim
B5	sim		B5		sim
B6	nao		B5		sim
